# InternVL2 LoRA SFT (rank=16 + early stopping) — VQA-RAD

LoRA rank **16** with **early stopping** on val `eval_loss` (patience=2, eval every epoch).

**Prerequisites**
1. Run `python3 scripts/build_train_json.py`
2. Base model at `models/OpenGVLab/InternVL2-8B`

**Outputs**
- `checkpoints/internvl2-vqa-lora-r16/adapter/` (best LoRA weights after early stopping)
- TensorBoard logs in `outputs/tb_logs_r16/`


In [1]:
import json
import math
import sys
from functools import partial
from pathlib import Path

import torch

PROJECT_ROOT = Path("/root/autodl-tmp/VQA").resolve()
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from internvl_sft_utils import (
    InternVLSFTDataset,
    InternVLTrainer,
    TrainConfig,
    apply_lora,
    build_training_arguments,
    build_trainer_callbacks,
    collate_internvl_batch,
    load_base_model,
    load_json_records,
    load_tokenizer,
)

MODEL_PATH = PROJECT_ROOT / "models/OpenGVLab/InternVL2-8B"
TRAIN_FILE = PROJECT_ROOT / "data/train_internvl.json"
VAL_FILE = PROJECT_ROOT / "data/val_internvl.json"
OUTPUT_DIR = PROJECT_ROOT / "checkpoints/internvl2-vqa-lora-r16"

EFFECTIVE_BATCH = 2 * 8  # per_device_batch_size * gradient_accumulation_steps
STEPS_PER_EPOCH = math.ceil(1570 / EFFECTIVE_BATCH)  # 98 steps/epoch

cfg = TrainConfig(
    project_root=PROJECT_ROOT,
    model_path=MODEL_PATH,
    train_file=TRAIN_FILE,
    val_file=VAL_FILE,
    output_dir=OUTPUT_DIR,
    num_epochs=5,
    per_device_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    max_num_patches=6,
    lora_rank=16,
    lora_alpha=32,
    train_projector=False,
    logging_dir=PROJECT_ROOT / "outputs/tb_logs_r16",
    eval_steps=STEPS_PER_EPOCH,
    save_steps=STEPS_PER_EPOCH,
    early_stopping_patience=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

print("Project root:", PROJECT_ROOT)
print("Model path:", MODEL_PATH)
print("Output dir:", OUTPUT_DIR)
print("LoRA rank/alpha:", cfg.lora_rank, cfg.lora_alpha)
print("Steps per epoch:", STEPS_PER_EPOCH)
print("Early stopping patience:", cfg.early_stopping_patience)


Project root: /root/autodl-tmp/VQA
Model path: /root/autodl-tmp/VQA/models/OpenGVLab/InternVL2-8B
Output dir: /root/autodl-tmp/VQA/checkpoints/internvl2-vqa-lora-r16
LoRA rank/alpha: 16 32
Steps per epoch: 99
Early stopping patience: 2


In [2]:
# Build SFT JSON if missing
if not TRAIN_FILE.exists() or not VAL_FILE.exists():
    !python3 {SCRIPTS_DIR / "build_train_json.py"}

train_records = load_json_records(TRAIN_FILE)
val_records = load_json_records(VAL_FILE)
print(f"train={len(train_records)}, val={len(val_records)}")
print("sample:", json.dumps(train_records[0], ensure_ascii=False, indent=2)[:800])


train=1570, val=224
sample: {
  "id": "test_0001",
  "image": "data/images/test_0001.jpg",
  "conversations": [
    {
      "from": "human",
      "value": "<image>\nYou are a radiology expert. This is a yes/no question about a medical image. Reply with only one word: 'yes' or 'no'.\nQuestion: is there airspace consolidation on the left side?"
    },
    {
      "from": "gpt",
      "value": "yes"
    }
  ],
  "question": "is there airspace consolidation on the left side?",
  "answer": "yes",
  "answer_type": "closed",
  "split": "train",
  "prompt_version": "v2"
}


In [3]:
tokenizer = load_tokenizer(MODEL_PATH)
model = load_base_model(MODEL_PATH, use_flash_attn=False)
model = apply_lora(
    model,
    rank=cfg.lora_rank,
    alpha=cfg.lora_alpha,
    dropout=cfg.lora_dropout,
    train_projector=cfg.train_projector,
)
model.cuda()

template_name = model.config.template
num_image_token = model.num_image_token
img_context_token_id = tokenizer.convert_tokens_to_ids("<IMG_CONTEXT>")

train_dataset = InternVLSFTDataset(
    train_records,
    project_root=PROJECT_ROOT,
    tokenizer=tokenizer,
    template_name=template_name,
    num_image_token=num_image_token,
    max_num_patches=cfg.max_num_patches,
)
val_dataset = InternVLSFTDataset(
    val_records,
    project_root=PROJECT_ROOT,
    tokenizer=tokenizer,
    template_name=template_name,
    num_image_token=num_image_token,
    max_num_patches=cfg.max_num_patches,
)

pad_token_id = tokenizer.pad_token_id
collate_fn = partial(collate_internvl_batch, pad_token_id=pad_token_id)

print("template:", template_name)
print("num_image_token:", num_image_token)
print("train samples:", len(train_dataset), "val samples:", len(val_dataset))


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
/root/miniconda3/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


FlashAttention2 is not installed.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

trainable params: 37,748,736 || all params: 7,775,531,008 || trainable%: 0.4855
Trainable parameter summary:
  LoRA (language_model): 37,748,736
  Projector (mlp1):      0  (enabled=False)
  Vision (should be 0):  0
  Total trainable:       37,748,736 / 8,113,114,112 (0.4653%)
template: internlm2-chat
num_image_token: 256
train samples: 1570 val samples: 224


trainable params: 18,874,368 || all params: 7,756,656,640 || trainable%: 0.2433
template: internlm2-chat
num_image_token: 256
train samples: 1570 val samples: 224


In [4]:
training_args = build_training_arguments(cfg)

trainer = InternVLTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn,
    img_context_token_id=img_context_token_id,
    callbacks=build_trainer_callbacks(cfg),
)

print("Effective batch size:", cfg.per_device_batch_size * cfg.gradient_accumulation_steps)
print("Eval/save every:", cfg.eval_steps, "steps (~1 epoch)")
print("Max epochs:", cfg.num_epochs, "| early stopping patience:", cfg.early_stopping_patience)
print("Output dir:", OUTPUT_DIR)


[RANK 0] Detected kernel version 4.19.90, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Effective batch size: 16
Eval/save every: 99 steps (~1 epoch)
Max epochs: 5 | early stopping patience: 2
Output dir: /root/autodl-tmp/VQA/checkpoints/internvl2-vqa-lora-r16


Effective batch size: 16
Output dir: /root/autodl-tmp/VQA/checkpoints/internvl2-vqa-lora


In [5]:
# Start training (run in foreground); early stopping may finish before num_epochs
train_result = trainer.train()
print(train_result)

# Save LoRA adapter offline (no HuggingFace Hub requests)
import os

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")

adapter_dir = OUTPUT_DIR / "adapter"
adapter_dir.mkdir(parents=True, exist_ok=True)

local_base = str(MODEL_PATH.resolve())
peft_model = model.language_model

# Point PEFT metadata to the local InternVL2 checkpoint before saving
if hasattr(peft_model, "peft_config"):
    for cfg_key in peft_model.peft_config:
        peft_model.peft_config[cfg_key].base_model_name_or_path = local_base

peft_model.save_pretrained(adapter_dir, safe_serialization=True)

# Force adapter_config.json to keep the local base model path
adapter_cfg_path = adapter_dir / "adapter_config.json"
if adapter_cfg_path.exists():
    adapter_cfg = json.loads(adapter_cfg_path.read_text(encoding="utf-8"))
    adapter_cfg["base_model_name_or_path"] = local_base
    adapter_cfg_path.write_text(
        json.dumps(adapter_cfg, indent=2, ensure_ascii=False) + "\n",
        encoding="utf-8",
    )

state = trainer.state
meta = {
    "base_model": local_base,
    "lora_rank": cfg.lora_rank,
    "lora_alpha": cfg.lora_alpha,
    "train_projector": cfg.train_projector,
    "learning_rate": cfg.learning_rate,
    "prompt_version": "v2",
    "train_file": str(TRAIN_FILE),
    "val_file": str(VAL_FILE),
    "train_samples": len(train_records),
    "val_samples": len(val_records),
    "adapter_dir": str(adapter_dir),
    "early_stopping_patience": cfg.early_stopping_patience,
    "load_best_model_at_end": cfg.load_best_model_at_end,
    "best_eval_loss": state.best_metric,
    "best_checkpoint": state.best_model_checkpoint,
    "global_step": state.global_step,
    "epoch": state.epoch,
}
(OUTPUT_DIR / "train_meta.json").write_text(
    json.dumps(meta, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)
print("Saved adapter ->", adapter_dir)
print("Best eval_loss:", state.best_metric)
print("Best checkpoint:", state.best_model_checkpoint)
print("Local base model:", local_base)


/root/miniconda3/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/root/miniconda3/compiler_compat/ld: warning: librt.so.1, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/root/miniconda3/compiler_compat/ld: warning: libpthread.so.0, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/root/miniconda3/compiler_compat/ld: warning: libstdc++.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/root/miniconda3/compiler_compat/ld: warning: libm.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/root/miniconda3/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `std::runtime_error::~runtime_error()@GLIBCXX_3.4'
/root/miniconda3/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `__gxx_personality_v0@CXXABI_1.3

Step,Training Loss,Validation Loss
99,0.688700,0.756416
198,0.462500,0.704894
297,0.166700,0.877902


/root/miniconda3/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/root/miniconda3/lib/python3.12/site-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/root/miniconda3/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can

SafetensorError: Error while serializing: I/O error: No space left on device (os error 28)

TrainOutput(global_step=294, training_loss=0.5065847793403937, metrics={'train_runtime': 5001.6946, 'train_samples_per_second': 0.942, 'train_steps_per_second': 0.059, 'total_flos': 6.017067773874835e+20, 'train_loss': 0.5065847793403937, 'epoch': 2.996178343949045})


'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /internlm/internlm2_5-7b-chat/resolve/main/config.json (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x7f18bd94b740>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: 1097b62f-682f-4534-8d48-f6199c21ac3c)')' thrown while requesting HEAD https://huggingface.co/internlm/internlm2_5-7b-chat/resolve/main/config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /internlm/internlm2_5-7b-chat/resolve/main/config.json (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x7f18bd94a450>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: 5a387060-64b4-4db1-9657-49b79e89967e)')' thrown while requesting HEAD https://huggingface.co/internlm/internlm2_5-7b-chat/resolve/main/config.json
Retrying in 

Saved adapter -> /root/autodl-tmp/VQA/checkpoints/internvl2-vqa-lora/adapter


/root/miniconda3/lib/python3.12/site-packages/peft/utils/other.py:611: UserWarning: Unable to fetch remote file due to the following error (MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /internlm/internlm2_5-7b-chat/resolve/main/config.json (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x7f195f1d62a0>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: fd3ab1d5-d3e1-4ad2-b514-88bbed79596c)') - silently ignoring the lookup for the file config.json in internlm/internlm2_5-7b-chat.
  warnings.warn(
/root/miniconda3/lib/python3.12/site-packages/peft/utils/save_and_load.py:195: UserWarning: Could not find a config file in internlm/internlm2_5-7b-chat - will assume that the vocabulary was not modified.
  warnings.warn(


In [ ]:
# Evaluate on val / test
ADAPTER_DIR = OUTPUT_DIR / "adapter"

!python3 {SCRIPTS_DIR / "eval_lora.py"} \
  --model-path {MODEL_PATH} \
  --lora-path {ADAPTER_DIR} \
  --test-file {PROJECT_ROOT / "data/vqa_rad_val.jsonl"} \
  --output-dir {PROJECT_ROOT / "outputs"} \
  --metrics-file lora_r16_val_metrics.json \
  --predictions-file lora_r16_val_predictions.jsonl

!python3 {SCRIPTS_DIR / "eval_lora.py"} \
  --model-path {MODEL_PATH} \
  --lora-path {ADAPTER_DIR} \
  --test-file {PROJECT_ROOT / "data/vqa_rad_test.jsonl"} \
  --output-dir {PROJECT_ROOT / "outputs"} \
  --metrics-file lora_r16_test_metrics.json \
  --predictions-file lora_r16_test_predictions.jsonl